# Chicago Taxi Trips — week of 11–17 September 2023
SUMMARY_PLACEHOLDER

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('figures/analysis', exist_ok=True)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
BLUE, ORANGE, GREY = '#2a78d6', '#eb6834', '#8a8985'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'figure.dpi': 100, 'axes.titlesize': 13, 'axes.titleweight': 'bold'})
raw = pd.read_csv('trips.csv', dtype={'pickup_census_tract': 'string', 'dropoff_census_tract': 'string'},
                  parse_dates=['trip_start_timestamp', 'trip_end_timestamp'])
print(raw.shape)
print(raw.dtypes)

(135768, 23)
trip_id                                  str
taxi_id                                  str
trip_start_timestamp          datetime64[us]
trip_end_timestamp            datetime64[us]
trip_seconds                         float64
trip_miles                           float64
pickup_census_tract                   string
dropoff_census_tract                  string
pickup_community_area                float64
dropoff_community_area               float64
fare                                 float64
tips                                 float64
tolls                                float64
extras                               float64
trip_total                           float64
payment_type                             str
company                                  str
pickup_centroid_latitude             float64
pickup_centroid_longitude            float64
pickup_centroid_location                 str
dropoff_centroid_latitude            float64
dropoff_centroid_longitude           float

In [2]:
print(raw.isna().sum())
print('duplicate trip_id:', raw.trip_id.duplicated().sum())
print(raw.trip_start_timestamp.min(), raw.trip_start_timestamp.max())
print(raw.describe().T)

trip_id                           0
taxi_id                           0
trip_start_timestamp              0
trip_end_timestamp                1
trip_seconds                     33
trip_miles                        0
pickup_census_tract           71404
dropoff_census_tract          73842
pickup_community_area          3815
dropoff_community_area        13794
fare                            324
tips                            324
tolls                           324
extras                          324
trip_total                      324
payment_type                      0
company                           0
pickup_centroid_latitude       3775
pickup_centroid_longitude      3775
pickup_centroid_location       3775
dropoff_centroid_latitude     12872
dropoff_centroid_longitude    12872
dropoff_centroid_location     12872
dtype: int64
duplicate trip_id: 0
2023-09-11 00:00:00 2023-09-17 23:45:00
                               count                        mean                  min             

In [3]:
print(raw.payment_type.value_counts(dropna=False))
print()
print(raw.company.value_counts(dropna=False).to_string())

payment_type
Credit Card    60291
Cash           40053
Mobile         18594
Prcard         11961
Unknown         4784
No Charge         54
Dispute           31
Name: count, dtype: int64

company
Flash Cab                               25950
Taxi Affiliation Services               25727
Sun Taxi                                15122
Taxicab Insurance Agency Llc            13914
City Service                            13747
Chicago Independents                     8004
5 Star Taxi                              6121
Medallion Leasin                         5003
Globe Taxi                               4772
Blue Ribbon Taxi Association             4390
Star North Taxi Management Llc           2815
Taxicab Insurance Agency, LLC            2379
Choice Taxi Association                  2022
Chicago City Taxi Association            1386
Choice Taxi Association Inc              1064
U Taxicab                                1048
Chicago Taxicab                           406
Patriot Taxi Dba Peace 

In [4]:
money = ['fare','tips','tolls','extras','trip_total']
nomoney = raw[money].isna().all(axis=1)
print('rows missing all money fields:', nomoney.sum(), '| any missing:', raw[money].isna().any(axis=1).sum())
print(raw[nomoney][['trip_seconds','trip_miles','payment_type','company']].describe(include='all').T)
print()
# pickup area missing but centroid present
odd = raw.pickup_community_area.isna() & raw.pickup_centroid_latitude.notna()
print('no community area but has centroid:', odd.sum())
print(raw[odd][['pickup_census_tract','pickup_centroid_latitude','pickup_centroid_longitude']].head())
print('trip_end missing:'); print(raw[raw.trip_end_timestamp.isna()].T.head(12))

rows missing all money fields: 324 | any missing: 324
              count unique                           top freq        mean          std  min     25%    50%      75%     max
trip_seconds  316.0    NaN                           NaN  NaN  1231.85443  1166.560076  2.0  398.75  727.0   1923.5  7278.0
trip_miles    324.0    NaN                           NaN  NaN    7.143765     8.608393  0.0  0.8775   2.32  12.9125   34.65
payment_type    324      2                          Cash  323         NaN          NaN  NaN     NaN    NaN      NaN     NaN
company         324     12  Taxicab Insurance Agency Llc  104         NaN          NaN  NaN     NaN    NaN      NaN     NaN

no community area but has centroid: 40
      pickup_census_tract  pickup_centroid_latitude  pickup_centroid_longitude
5285          17031770700                 41.982775                 -87.877305
5912          17031770700                 41.982775                 -87.877305
12033         17031770700                 41.9827

In [ ]:
d = raw[~nomoney].copy()
d['parts'] = d[['fare','tips','tolls','extras']].sum(axis=1)
d['diff'] = (d.trip_total - d.parts).round(2)
print('total != sum of parts (>0.01):', (d['diff'].abs() > 0.01).sum())
print(d['diff'].describe())
print(d.loc[d['diff'].abs() > 0.01, ['fare','tips','tolls','extras','trip_total','diff','payment_type']].sort_values('diff').head(8))
print(d.loc[d['diff'].abs() > 0.01, ['fare','tips','tolls','extras','trip_total','diff','payment_type']].sort_values('diff').tail(8))
print('diff value counts (top):'); print(d['diff'].round(2).value_counts().head(8))

In [ ]:
print('trip_seconds==0:', (raw.trip_seconds == 0).sum(), '| NaN:', raw.trip_seconds.isna().sum(), '| >6h:', (raw.trip_seconds > 6*3600).sum(), '| >24h:', (raw.trip_seconds >= 86400).sum())
print('trip_miles==0:', (raw.trip_miles == 0).sum(), '| >100:', (raw.trip_miles > 100).sum(), '| >50:', (raw.trip_miles > 50).sum())
print('fare==0 (non-missing):', (d.fare == 0).sum(), '| fare>200:', (d.fare > 200).sum(), '| fare>100:', (d.fare > 100).sum())
print('trip_total==0:', (d.trip_total == 0).sum(), '| >200:', (d.trip_total > 200).sum(), '| tolls>50:', (d.tolls > 50).sum(), '| extras>50:', (d.extras > 50).sum())
print('negative any:', (raw[['trip_seconds','trip_miles']+money] < 0).sum().sum())
print()
print('quantiles'); print(raw[['trip_seconds','trip_miles','fare','trip_total']].quantile([.001,.01,.5,.99,.999,.9999]).round(2))